In [1]:
import time
notebook_start_time = time.time()

NOTICE: Code is optimized for the A100 GPU

# 1. Installing, importing libraries and making configurations

In [2]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 97.5 MB/s eta 0:00:00


In [3]:
!uv pip install torch numpy matplotlib lightning huggingface_hub triton pandas optuna optuna-integration[pytorch_lightning]

Using Python 3.12.13 environment at: /usr
Resolved 79 packages in 743ms
Prepared 7 packages in 117ms
Installed 7 packages in 12ms
 + colorlog==6.10.1
 + lightning==2.6.1
 + lightning-utilities==0.15.3
 + optuna==4.8.0
 + optuna-integration==4.8.0
 + pytorch-lightning==2.6.1
 + torchmetrics==1.9.0


In [4]:
!uv pip install mamba-ssm --no-build-isolation-package mamba-ssm

Using Python 3.12.13 environment at: /usr
Resolved 56 packages in 432ms
Prepared 1 package in 26ms
Installed 1 package in 2ms
Prepared 1 package without build isolation in 14.26s
Installed 1 package in 1ms
 + mamba-ssm==2.3.1
 + ninja==1.13.0


In [5]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


import lightning as L
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download, login
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split

from mamba_ssm import Mamba

import optuna
from optuna.integration import PyTorchLightningPruningCallback

from mamba_ssm.ops.selective_scan_interface import selective_scan_fn

In [6]:
NUM_WORKERS = 4

NUM_EPOCHS = 10

# 2. Models developed

In [7]:
def label_smoothed_bce_loss(logits, labels, eps=0.1):
    ce_hard = F.cross_entropy(logits, labels)
    log_probs = F.log_softmax(logits, dim=-1)
    ce_soft = -log_probs.mean(dim=-1).mean()
    return (1.0 - eps) * ce_hard + eps * ce_soft

In [8]:
def get_optimizer(model, lr=1e-3, base_wd=0.01, delta_wd=1e-3):
    delta_params = []
    other_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if 'delta_fixed' in name:
            delta_params.append(param)
        else:
            other_params.append(param)
    param_groups = [
        {'params': other_params, 'weight_decay': base_wd,  'name': 'default'},
        {'params': delta_params, 'weight_decay': delta_wd, 'name': 'delta_fixed'},
    ]
    return torch.optim.AdamW(param_groups, lr=lr)

In [9]:
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight

In [10]:
class ResidualBlock(nn.Module):
    def __init__(self, d_model, N, expand=2, **kwargs):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.ssm  = FixedSSMBlock(d_model=d_model, d_state=N, expand=expand)

    def forward(self, x):
        return x + self.ssm(self.norm(x))

In [11]:
class FixedSSMBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.d_inner = d_model * expand
        self.d_state = d_state

        self.in_proj  = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner,
            kernel_size=d_conv, padding=d_conv - 1, groups=self.d_inner,
        )

        A = torch.arange(1, d_state + 1, dtype=torch.float32)
        A = A.unsqueeze(0).expand(self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        # FIXED (non-input-dependent) B, C, delta
        self.B_fixed = nn.Parameter(torch.randn(d_state) * 0.02)
        self.C_fixed = nn.Parameter(torch.randn(d_state) * 0.02)
        self.delta_fixed = nn.Parameter(torch.zeros(self.d_inner))

    def forward(self, x):
        batch, length, _ = x.shape

        xz = self.in_proj(x).transpose(1, 2)
        x_inner, z = xz.chunk(2, dim=1)

        x_inner = self.conv1d(x_inner)[..., :length]
        x_inner = F.silu(x_inner)

        A = -torch.exp(self.A_log)
        delta = (F.softplus(self.delta_fixed)
                 .unsqueeze(0).unsqueeze(-1)
                 .expand(batch, -1, length).contiguous())
        B = (self.B_fixed
             .unsqueeze(0).unsqueeze(-1)
             .expand(batch, -1, length).contiguous())
        C = (self.C_fixed
             .unsqueeze(0).unsqueeze(-1)
             .expand(batch, -1, length).contiguous())

        delta, B, C = delta.to(x_inner.dtype), B.to(x_inner.dtype), C.to(x_inner.dtype)

        y = selective_scan_fn(
            x_inner, delta, A, B, C,
            D=self.D, z=z, delta_softplus=False,
        )
        return self.out_proj(y.transpose(1, 2))

In [12]:
class MambaClassifier(L.LightningModule):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 128,
        n_layers: int = 2,
        N: int = 16,
        num_classes: int = 2,
        lr: float = 1e-3,
        label_smoothing: float = 0.0,
        delta_wd: float = 0.0,
        expand: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.label_smoothing = label_smoothing
        self.delta_wd = delta_wd
        self.num_classes = num_classes

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            ResidualBlock(d_model, N, expand=expand) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

        # Compile Mamba layers for fused Triton kernels (replaces causal-conv1d)
        for i, layer in enumerate(self.layers):
            self.layers[i] = torch.compile(layer)

    def forward(self, input_ids, attention_mask=None):
        x = self.embedding(input_ids)               # (B, L, D)

        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)

        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            x = x.mean(dim=1)

        return self.classifier(x)

    def _shared_step(self, batch, stage):
        logits = self(batch['input_ids'], batch['attention_mask'])
        labels = batch['labels']

        if self.label_smoothing > 0:
            loss = label_smoothed_bce_loss(logits, labels, eps=self.label_smoothing)
        else:
            loss = F.cross_entropy(logits, labels)

        preds = logits.argmax(dim=-1)
        acc = (preds == labels).float().mean()

        self.log(f'{stage}_loss', loss, prog_bar=True)
        self.log(f'{stage}_acc',  acc,  prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, 'train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, 'val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, 'test')

    def configure_optimizers(self):
        if self.delta_wd > 0:
            return get_optimizer(
                self, lr=self.lr, base_wd=0.01, delta_wd=self.delta_wd
            )
        return torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=0.01)

# 3. Data and running each model in its benchmark

## 3.0 Data and others

In [13]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt', 'cifar10_sequential_lra.pt']

for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


listops.pt:   0%|          | 0.00/175M [00:00<?, ?B/s]

imdb_lra.pt:   0%|          | 0.00/132M [00:00<?, ?B/s]

acl_retrieval_lra.pt:   0%|          | 0.00/274M [00:00<?, ?B/s]

cifar10_sequential_lra.pt:   0%|          | 0.00/124M [00:00<?, ?B/s]

In [14]:
def make_trainer():
    return L.Trainer(
        max_epochs=NUM_EPOCHS,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        logger=CSVLogger(save_dir="logs", name="lra-benchmarks"),
    )

In [15]:
def make_hpo_trainer(max_epochs=NUM_EPOCHS):
    """Lightweight trainer for HPO — no checkpointing, no logging."""
    return L.Trainer(
        max_epochs=max_epochs,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        devices=1,
        precision='bf16-mixed',
        enable_checkpointing=False,
        logger=False,
    )

In [16]:
COMBINATIONS = {
    "combination_1": {"d_model": 64,  "n_layers": 4, "N": 8,  "lr": 5e-4},
    "combination_2": {"d_model": 128, "n_layers": 2, "N": 16, "lr": 5e-4},
}

def run_both_combinations(dataset_name, DatasetClass, data_dict, batch_sizes):
    """
    batch_sizes: dict like {"combination_1": 8, "combination_2": 4}
    """
    train_labels = [item[1] for item in data_dict['train'].data]
    num_classes = max(train_labels) + 1

    results = {}

    for combo_name, cfg in COMBINATIONS.items():
        bs = batch_sizes[combo_name]
        print(f"\n{'='*50}\n{dataset_name} — {combo_name}: {cfg} | batch_size={bs}\n{'='*50}")

        train_dataset = DatasetClass(data_dict['train'].data)
        val_dataset   = DatasetClass(data_dict['val'].data)

        train_loader = DataLoader(train_dataset, batch_size=bs, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
        val_loader   = DataLoader(val_dataset, batch_size=bs,
                                  num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

        model = MambaClassifier(
            vocab_size=data_dict['vocab_size'],
            d_model=cfg['d_model'], n_layers=cfg['n_layers'], N=cfg['N'],
            num_classes=num_classes, lr=cfg['lr'],
        )

        total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parameters: {total_params}")

        trainer = make_trainer()
        trainer.fit(model, train_loader, val_loader)

        test_dataset = DatasetClass(data_dict['test'].data)
        test_loader  = DataLoader(test_dataset, batch_size=32, num_workers=NUM_WORKERS,
                                  pin_memory=True, persistent_workers=True)
        trainer.test(model, test_loader)

        results[combo_name] = {
            "config": {**cfg, "batch_size": bs},
            "params": total_params,
            "val_acc": trainer.callback_metrics.get("val_acc", None),
            "test_acc": trainer.callback_metrics.get("test_acc", None),
        }

    return results

## 3.1 ListOpsDataset

In [17]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [18]:
data_listops = torch.load('listops.pt', weights_only=False)

In [19]:
results_listops = run_both_combinations(
    "ListOps", ListOpsDataset, data_listops,
    batch_sizes={"combination_1": 8, "combination_2": 4},
)


ListOps — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005} | batch_size=8


INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/

Parameters: 112266


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │  1.1 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

...

/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.5099999904632568     │
│         test_loss         │    1.3080254793167114     │
└───────────────────────────┴───────────────────────────┘

## 3.2 IMDbDataset

In [20]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [21]:
data_imdb = torch.load('imdb_lra.pt', weights_only=False)

In [22]:
results_imdb = run_both_combinations(
    "IMDb", IMDbDataset, data_imdb,
    batch_sizes={"combination_1": 8, "combination_2": 4},
)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)



IMDb — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005} | batch_size=8
Parameters: 127106


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.843720018863678     │
│         test_loss         │     0.536344587802887     │
└───────────────────────────┴───────────────────────────┘

## 3.3 ACLRetricalDataset

In [23]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [24]:
data_retrieval = torch.load('acl_retrieval_lra.pt', weights_only=False)

In [25]:
results_retrieval = run_both_combinations(
    "Retrieval", ACLRetrievalDataset, data_retrieval,
    batch_sizes={"combination_1": 8, "combination_2": 4},
)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/


Retrieval — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005} | batch_size=8
Parameters: 127106


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.3 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4970000088214874     │
│         test_loss         │     0.706613302230835     │
└───────────────────────────┴───────────────────────────┘

## 3.4 CIFAR10SequentialDataset

In [26]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Handle both dict and tuple formats
        if isinstance(item, dict):
            tokens = item['input_ids']
            label = item['labels']
        else:
            tokens = item[0]
            label = item[-1]

        # Convert tensors to lists if needed (for padding logic)
        if isinstance(tokens, torch.Tensor):
            tokens = tokens.tolist()
        if isinstance(label, torch.Tensor):
            label = label.item()

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [27]:
data_cifar10_sequential = torch.load('cifar10_sequential_lra.pt', weights_only=False)

In [28]:
results_cifar = run_both_combinations(
    "CIFAR-10", CIFAR10SequentialDataset, data_cifar10_sequential,
    batch_sizes={"combination_1": 16, "combination_2": 8},
)

INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https:/


CIFAR-10 — combination_1: {'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005} | batch_size=16
Parameters: 127626


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ embedding  │ Embedding  │ 16.4 K │ train │     0 │
│ 1 │ layers     │ ModuleList │  106 K │ train │     0 │
│ 2 │ norm_f     │ RMSNorm    │     64 │ train │     0 │
│ 3 │ classifier │ Sequential │  4.8 K │ train │     0 │
└───┴────────────┴────────────┴────────┴───────┴───────┘

Trainable params: 127 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 127 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Output()

INFO: `Trainer.fit` stopped: `max_epochs=10` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.35499998927116394    │
│         test_loss         │    2.2107396125793457     │
└───────────────────────────┴───────────────────────────┘

# 4. Seeing results

In [29]:
print("\n" + "=" * 70)
print("  RESULTS SUMMARY — BOTH COMBINATIONS")
print("=" * 70)

for name, results in [
    ("ListOps",   results_listops),
    ("IMDb",      results_imdb),
    ("Retrieval", results_retrieval),
    ("CIFAR-10",  results_cifar),
]:
    print(f"\n  {name}")
    for combo_name, r in results.items():
        print(f"    {combo_name}: val_acc={r['val_acc']}  test_acc={r['test_acc']}  params={r['params']}  config={r['config']}")

print("\n" + "=" * 70)


  RESULTS SUMMARY — BOTH COMBINATIONS

  ListOps
    combination_1: val_acc=None  test_acc=0.4375  params=112266  config={'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
    combination_2: val_acc=None  test_acc=0.5099999904632568  params=228810  config={'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}

  IMDb
    combination_1: val_acc=None  test_acc=0.8508800268173218  params=127106  config={'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
    combination_2: val_acc=None  test_acc=0.843720018863678  params=258498  config={'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}

  Retrieval
    combination_1: val_acc=None  test_acc=0.4945000112056732  params=127106  config={'d_model': 64, 'n_layers': 4, 'N': 8, 'lr': 0.0005, 'batch_size': 8}
    combination_2: val_acc=None  test_acc=0.4970000088214874  params=258498  config={'d_model': 128, 'n_layers': 2, 'N': 16, 'lr': 0.0005, 'batch_size': 4}

  CIFAR-10


In [30]:
notebook_end_time = time.time()
total_time = notebook_end_time - notebook_start_time

# Convert to minutes and seconds for easier reading
minutes = int(total_time // 60)
seconds = int(total_time % 60)
print(f"Total 'Run All' execution time: {minutes} minutes and {seconds} seconds")

Total 'Run All' execution time: 223 minutes and 14 seconds
